# PS-1 LSTM + PS-3 Seq2Seq — Intelligent Customer Support Ticket System

This notebook covers the two parts of the project that need a GPU:

| Part | Problem statement | What it produces |
|---|---|---|
| **A** | PS-1 — intent classification with an LSTM | A score to compare against your TF-IDF baseline |
| **B** | PS-3 — automated response generation (Seq2Seq + attention) | Generated replies, BLEU / ROUGE, sample outputs |

### Before you run anything

1. **Runtime → Change runtime type → T4 GPU.** People routinely train for an hour on CPU without noticing.
2. Run `04_export_for_colab.py` locally and upload `colab_ps1.csv` and `colab_ps3.csv` to Google Drive, into a folder called `support_ticket_project`.
3. Every epoch checkpoints to Drive. Free Colab kills sessions without warning — an unsaved 40-minute run is gone.

### What to expect (so you are not surprised)

The LSTM in Part A will probably **not** beat your TF-IDF LinearSVC. That is a normal, reportable result: with ~200k short texts and rule-generated labels, a linear model on n-grams is very hard to beat. Report it honestly rather than tuning until deep learning "wins".

Part B will produce fluent but **generic** replies — lots of "sorry about that, please DM us". That is the known failure mode of vanilla Seq2Seq on support data, not a bug in your code. The notebook measures it explicitly so you can write about it.

## 0. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!! No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/support_ticket_project'
import os
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
print(os.listdir(DRIVE))

In [ ]:
!pip install -q rouge-score
import nltk; nltk.download('punkt', quiet=True)

In [ ]:
import random, math, time
from collections import Counter

import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reserved token ids. PAD must be 0 so that padding_idx=0 works everywhere.
PAD, UNK, SOS, EOS = 0, 1, 2, 3
print(DEV)

### The vocabulary

TF-IDF handed us a matrix; a neural network needs **integer sequences** instead. So we build a
vocabulary: every word that appears at least `min_freq` times gets an id, capped at `max_size`.

Two caps that matter:

- **`max_size=20000`** — the embedding layer is `vocab × emb_dim` floats. A 100k vocabulary is
  mostly typos and one-off usernames, and it quadruples memory for nothing.
- **`min_freq=2`** — a word seen once cannot be learned from, it can only be memorised.

Anything outside the vocabulary becomes `<unk>`. That is not a failure — it is the model learning
to cope with words it has never seen, which is what will happen in production.

In [ ]:
def tokenize(t):
    return str(t).lower().split()

class Vocab:
    def __init__(self, texts, max_size=20000, min_freq=2):
        counts = Counter(w for t in texts for w in tokenize(t))
        keep = [w for w, n in counts.most_common() if n >= min_freq][: max_size - 4]
        self.itos = ['<pad>', '<unk>', '<sos>', '<eos>'] + keep
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text, maxlen, sos=False, eos=False):
        '''Text -> fixed-length id list, plus the true (unpadded) length.'''
        ids = [self.stoi.get(w, UNK) for w in tokenize(text)][: maxlen - int(sos) - int(eos)]
        if sos: ids = [SOS] + ids
        if eos: ids = ids + [EOS]
        length = len(ids)
        return ids + [PAD] * (maxlen - length), length

    def decode(self, ids):
        return ' '.join(self.itos[i] for i in ids if i > EOS)

---
# PART A — PS-1: intent classification with an LSTM

**Why bother, when TF-IDF already scored 0.97?**

Because TF-IDF throws away word order. "not working" and "working not" are identical to it —
the `ngram_range=(1,2)` patched over that, but only for adjacent pairs. An LSTM reads the sequence
left to right and carries state, so in principle it can capture longer dependencies.

In practice, on short texts with rule-generated labels, it usually doesn't win. **That comparison is
the deliverable**, not a particular winner.

In [ ]:
MAXLEN_CLS = 40          # tweets are short; 40 tokens covers ~99% of them
EMB_CLS    = 200
HID_CLS    = 256
BATCH_CLS  = 128
EPOCHS_CLS = 6

df1 = pd.read_csv(f'{DRIVE}/colab_ps1.csv').dropna(subset=['clean_text', 'intent'])
print(f'{len(df1):,} rows')
print(df1['intent'].value_counts().to_string())

classes = sorted(df1['intent'].unique())
cls2id = {c: i for i, c in enumerate(classes)}
df1['y'] = df1['intent'].map(cls2id)

Xtr, Xte, ytr, yte = train_test_split(
    df1['clean_text'].values, df1['y'].values,
    test_size=0.2, random_state=SEED, stratify=df1['y'].values)
print(f'train {len(Xtr):,}   test {len(Xte):,}')

In [ ]:
# Vocab is built on the TRAINING TEXTS ONLY.
# Building it on everything would leak test vocabulary into the model - a subtle
# but real form of cheating that inflates your score.
vocab_cls = Vocab(Xtr, max_size=20000, min_freq=2)
print('vocab size:', len(vocab_cls))

class ClsDataset(Dataset):
    def __init__(self, texts, labels, vocab, maxlen):
        enc = [vocab.encode(t, maxlen) for t in texts]
        self.x = [e[0] for e in enc]
        self.len = [max(e[1], 1) for e in enc]   # min 1: a fully-<unk> row still needs length>=1
        self.y = labels

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return (torch.tensor(self.x[i]), torch.tensor(self.len[i]), torch.tensor(self.y[i]))

tr_dl = DataLoader(ClsDataset(Xtr, ytr, vocab_cls, MAXLEN_CLS), batch_size=BATCH_CLS, shuffle=True)
te_dl = DataLoader(ClsDataset(Xte, yte, vocab_cls, MAXLEN_CLS), batch_size=BATCH_CLS)

### The model

```
tokens -> Embedding -> BiLSTM -> concat(final forward, final backward) -> Dropout -> Linear -> 5 scores
```

- **Bidirectional** — reads the text forwards *and* backwards. For classification you have the whole
  message up front, so there is no reason to only look one way.
- **`pack_padded_sequence`** — this is the important line. Without it the LSTM processes your `<pad>`
  tokens as if they were real words, and a short message ends up with its meaning diluted by 30 steps
  of padding. Packing tells PyTorch exactly where each sequence really ends.
- **`padding_idx=PAD`** — keeps the pad embedding fixed at zero.

In [ ]:
class IntentLSTM(nn.Module):
    def __init__(self, vocab_size, n_classes, emb=EMB_CLS, hid=HID_CLS, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hid * 2, n_classes)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h = torch.cat([h[-2], h[-1]], dim=1)     # final forward + final backward state
        return self.fc(self.drop(h))

model_cls = IntentLSTM(len(vocab_cls), len(classes)).to(DEV)
print(model_cls)
print('parameters:', f'{sum(p.numel() for p in model_cls.parameters()):,}')

In [ ]:
# Class weights do the same job class_weight="balanced" did in sklearn: stop the
# model quietly ignoring Account, which is only ~6% of rows.
counts = np.bincount(ytr, minlength=len(classes))
weights = torch.tensor(len(ytr) / (len(classes) * counts), dtype=torch.float32, device=DEV)
print(dict(zip(classes, weights.cpu().numpy().round(2))))

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model_cls.parameters(), lr=1e-3)

def evaluate(model, dl):
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for x, l, y in dl:
            out = model(x.to(DEV), l)
            preds += out.argmax(1).cpu().tolist(); trues += y.tolist()
    return trues, preds

best_f1 = 0.0
for epoch in range(1, EPOCHS_CLS + 1):
    model_cls.train(); total = 0; t0 = time.time()
    for x, l, y in tr_dl:
        optimizer.zero_grad()
        loss = criterion(model_cls(x.to(DEV), l), y.to(DEV))
        loss.backward()
        nn.utils.clip_grad_norm_(model_cls.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    trues, preds = evaluate(model_cls, te_dl)
    f1 = f1_score(trues, preds, average='macro')
    print(f'epoch {epoch}  loss {total/len(tr_dl):.4f}  macro-F1 {f1:.4f}  ({time.time()-t0:.0f}s)')

    # checkpoint every epoch, keep the best
    if f1 > best_f1:
        best_f1 = f1
        torch.save({'model': model_cls.state_dict(), 'itos': vocab_cls.itos,
                    'classes': classes, 'macro_f1': f1},
                   f'{DRIVE}/checkpoints/ps1_lstm_best.pt')
        print('   saved new best')

In [ ]:
trues, preds = evaluate(model_cls, te_dl)
print(classification_report(trues, preds, target_names=classes, zero_division=0))
print('LSTM macro-F1     :', round(f1_score(trues, preds, average='macro'), 4))
print('TF-IDF LinearSVC  : 0.971   <- from script 03, on the sample')
print()
print('Whichever wins, say so plainly in the report and explain why.')
print('A linear model on n-grams beating an LSTM on short rule-labelled text')
print('is a legitimate finding, not a mistake.')

---
# PART B — PS-3: response generation with Seq2Seq + attention

### How this differs from Part A

Classification maps a sequence to **one label**. Generation maps a sequence to **another sequence**,
one word at a time, where each word depends on the ones already produced.

The architecture:

```
customer message -> ENCODER (BiLSTM) -> context vectors
                                            |
                              ATTENTION (which input words matter now?)
                                            |
                    DECODER (LSTM) -> "sorry" -> "about" -> "that" -> <eos>
```

**Why attention.** Without it, the encoder must compress the entire message into one fixed vector and
the decoder reads only that — a bottleneck that gets worse the longer the input. Attention lets the
decoder look back at *all* encoder positions at every step and weight them, so generating "refund" can
focus on the part of the message that mentioned money.

**Teacher forcing.** During training we sometimes feed the decoder the *correct* previous word instead
of its own prediction. Without it, one early mistake derails the whole sequence and the model never
learns. With it always on, the model never practises recovering from its own errors. So we use it
probabilistically and decay it across epochs.

In [ ]:
MAXLEN_SRC = 40
MAXLEN_TGT = 40
EMB_S2S    = 256
HID_S2S    = 512
BATCH_S2S  = 128
EPOCHS_S2S = 8

df3 = pd.read_csv(f'{DRIVE}/colab_ps3.csv').dropna()
print(f'{len(df3):,} pairs')
df3.head(3)

In [ ]:
src_tr, src_te, tgt_tr, tgt_te = train_test_split(
    df3['customer_text'].values, df3['response_text'].values,
    test_size=0.1, random_state=SEED)

# ONE shared vocabulary for questions and answers. They are the same language and
# share most words; two vocabularies would double the embedding memory for little gain.
vocab_s2s = Vocab(list(src_tr) + list(tgt_tr), max_size=20000, min_freq=2)
print('shared vocab:', len(vocab_s2s))

class S2SDataset(Dataset):
    def __init__(self, src, tgt, vocab):
        self.s = [vocab.encode(t, MAXLEN_SRC) for t in src]
        self.t = [vocab.encode(t, MAXLEN_TGT, sos=True, eos=True) for t in tgt]

    def __len__(self):
        return len(self.s)

    def __getitem__(self, i):
        (si, sl), (ti, _) = self.s[i], self.t[i]
        return torch.tensor(si), torch.tensor(max(sl, 1)), torch.tensor(ti)

s2s_tr = DataLoader(S2SDataset(src_tr, tgt_tr, vocab_s2s), batch_size=BATCH_S2S, shuffle=True)
s2s_te = DataLoader(S2SDataset(src_te, tgt_te, vocab_s2s), batch_size=BATCH_S2S)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, V, emb, hid):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=PAD)
        self.rnn = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        # the decoder is single-direction, so squash the two encoder directions into one
        self.fh = nn.Linear(hid * 2, hid)
        self.fcell = nn.Linear(hid * 2, hid)

    def forward(self, x, lengths):
        e = self.emb(x)
        p = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, c) = self.rnn(p)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        h = torch.tanh(self.fh(torch.cat([h[-2], h[-1]], 1))).unsqueeze(0)
        c = torch.tanh(self.fcell(torch.cat([c[-2], c[-1]], 1))).unsqueeze(0)
        return out, (h, c)

class Attention(nn.Module):
    '''Bahdanau (additive) attention: score every encoder position against the
    decoder's current state, softmax the scores into weights, and use them to
    build a weighted summary of the input.'''
    def __init__(self, hid):
        super().__init__()
        self.W = nn.Linear(hid * 2 + hid, hid)
        self.v = nn.Linear(hid, 1, bias=False)

    def forward(self, dec_h, enc_out, mask):
        T = enc_out.size(1)
        d = dec_h.repeat(T, 1, 1).transpose(0, 1)
        e = self.v(torch.tanh(self.W(torch.cat([d, enc_out], 2)))).squeeze(2)
        e = e.masked_fill(~mask, -1e9)     # never attend to padding
        return torch.softmax(e, dim=1)

class Decoder(nn.Module):
    def __init__(self, V, emb, hid):
        super().__init__()
        self.emb = nn.Embedding(V, emb, padding_idx=PAD)
        self.att = Attention(hid)
        self.rnn = nn.LSTM(emb + hid * 2, hid, batch_first=True)
        self.fc = nn.Linear(hid * 3 + emb, V)

    def forward(self, token, hidden, enc_out, mask):
        e = self.emb(token).unsqueeze(1)
        a = self.att(hidden[0][-1], enc_out, mask).unsqueeze(1)
        ctx = torch.bmm(a, enc_out)
        out, hidden = self.rnn(torch.cat([e, ctx], 2), hidden)
        pred = self.fc(torch.cat([out.squeeze(1), ctx.squeeze(1), e.squeeze(1)], 1))
        return pred, hidden

class Seq2Seq(nn.Module):
    def __init__(self, V, emb=EMB_S2S, hid=HID_S2S):
        super().__init__()
        self.enc = Encoder(V, emb, hid)
        self.dec = Decoder(V, emb, hid)
        self.V = V

    def forward(self, src, slen, tgt, tf_ratio=0.5):
        B, T = tgt.shape
        enc_out, hidden = self.enc(src, slen)
        mask = (src != PAD)
        outputs = torch.zeros(B, T, self.V, device=src.device)
        token = tgt[:, 0]                       # <sos>
        for t in range(1, T):
            pred, hidden = self.dec(token, hidden, enc_out, mask)
            outputs[:, t] = pred
            token = tgt[:, t] if random.random() < tf_ratio else pred.argmax(1)
        return outputs

    @torch.no_grad()
    def generate(self, src, slen, maxlen=MAXLEN_TGT):
        '''Greedy decoding: always take the highest-probability next word.'''
        enc_out, hidden = self.enc(src, slen)
        mask = (src != PAD)
        token = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
        done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
        res = []
        for _ in range(maxlen):
            pred, hidden = self.dec(token, hidden, enc_out, mask)
            token = pred.argmax(1)
            token = torch.where(done, torch.full_like(token, PAD), token)
            done = done | (token == EOS)
            res.append(token)
            if done.all():
                break
        return torch.stack(res, 1)

s2s = Seq2Seq(len(vocab_s2s)).to(DEV)
print('parameters:', f'{sum(p.numel() for p in s2s.parameters()):,}')

In [ ]:
opt_s2s = torch.optim.Adam(s2s.parameters(), lr=1e-3)
# ignore_index=PAD: do not penalise the model for what it emits after the real
# answer has ended. Without this, loss is dominated by predicting padding.
crit_s2s = nn.CrossEntropyLoss(ignore_index=PAD)

def val_loss():
    s2s.eval(); tot = 0
    with torch.no_grad():
        for src, sl, tgt in s2s_te:
            out = s2s(src.to(DEV), sl, tgt.to(DEV), tf_ratio=0.0)   # no teacher forcing at eval
            tot += crit_s2s(out[:, 1:].reshape(-1, out.size(-1)),
                            tgt[:, 1:].reshape(-1).to(DEV)).item()
    return tot / len(s2s_te)

best_val = float('inf')
for epoch in range(1, EPOCHS_S2S + 1):
    # decay teacher forcing 0.9 -> 0.5 so the model gradually learns to stand on its own
    tf_ratio = max(0.5, 0.9 - 0.05 * (epoch - 1))
    s2s.train(); tot = 0; t0 = time.time()
    for src, sl, tgt in s2s_tr:
        opt_s2s.zero_grad()
        out = s2s(src.to(DEV), sl, tgt.to(DEV), tf_ratio=tf_ratio)
        loss = crit_s2s(out[:, 1:].reshape(-1, out.size(-1)),
                        tgt[:, 1:].reshape(-1).to(DEV))
        loss.backward()
        nn.utils.clip_grad_norm_(s2s.parameters(), 1.0)   # LSTMs explode without this
        opt_s2s.step()
        tot += loss.item()
    vl = val_loss()
    print(f'epoch {epoch}  train {tot/len(s2s_tr):.4f}  val {vl:.4f}  '
          f'ppl {math.exp(min(vl, 20)):.1f}  tf {tf_ratio:.2f}  ({time.time()-t0:.0f}s)')

    if vl < best_val:
        best_val = vl
        torch.save({'model': s2s.state_dict(), 'itos': vocab_s2s.itos, 'val_loss': vl},
                   f'{DRIVE}/checkpoints/ps3_seq2seq_best.pt')
        print('   saved new best')

### Look at the output before you trust the metric

Perplexity going down does not mean the replies are useful. Read them.

In [ ]:
s2s.eval()
src, sl, tgt = next(iter(s2s_te))
gen = s2s.generate(src[:12].to(DEV), sl[:12])

for i in range(12):
    print('Q  :', vocab_s2s.decode(src[i].tolist()))
    print('REF:', vocab_s2s.decode(tgt[i].tolist()))
    print('GEN:', vocab_s2s.decode(gen[i].cpu().tolist()))
    print('-' * 90)

In [ ]:
# Generate over the whole test set once, then score everything from it.
refs, hyps = [], []
s2s.eval()
for src, sl, tgt in s2s_te:
    g = s2s.generate(src.to(DEV), sl)
    for r, h in zip(tgt.tolist(), g.cpu().tolist()):
        refs.append(vocab_s2s.decode(r))
        hyps.append(vocab_s2s.decode(h))
print(len(hyps), 'generated')

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# BLEU: n-gram overlap with the reference, precision-oriented.
# Smoothing is required - short tweets often share no 4-gram at all, and an
# unsmoothed zero anywhere zeroes the whole corpus score.
sm = SmoothingFunction().method4
bleu = corpus_bleu([[r.split()] for r in refs], [h.split() for h in hyps],
                   smoothing_function=sm)
print(f'corpus BLEU-4 : {bleu:.4f}')

# ROUGE-L: longest common subsequence, recall-oriented. Slow, so score a subset.
rs = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
sub = random.sample(range(len(refs)), min(2000, len(refs)))
r1 = np.mean([rs.score(refs[i], hyps[i])['rouge1'].fmeasure for i in sub])
rl = np.mean([rs.score(refs[i], hyps[i])['rougeL'].fmeasure for i in sub])
print(f'ROUGE-1 F     : {r1:.4f}')
print(f'ROUGE-L F     : {rl:.4f}')

### The diagnostic that matters most

BLEU rewards a model that always emits the safest, most common reply. Support data is full of
"sorry about that, please DM us", so a model that says *only* that can score respectably while being
useless. The cell below measures that directly: how many distinct replies did it actually produce?

**Report this number.** A low diversity ratio alongside a decent BLEU is a much more sophisticated
finding than BLEU alone, and it is the honest description of what vanilla Seq2Seq does to this dataset.

In [ ]:
uniq = len(set(hyps))
print(f'unique generations : {uniq:,} / {len(hyps):,}  ({uniq/len(hyps):.1%})')
print('\nmost common generated replies:')
for text, n in Counter(hyps).most_common(10):
    print(f'  {n:5d}  {text[:90]}')

---
## What to write up

**PS-1.** Report both scores side by side — TF-IDF LinearSVC vs LSTM — and say which won and why you
think so. If the linear model wins, the explanation is that the texts are short, the labels came from
keyword rules, and n-grams already capture nearly all the signal there is. That is a real finding.

**PS-3.** Report BLEU, ROUGE-1, ROUGE-L, validation perplexity, **and the diversity ratio**. Include
five or six sample generations — good and bad. Then discuss the generic-response problem and what
would address it: coverage or diversity-promoting loss, beam search with a length penalty, retrieval
instead of generation, or a pretrained transformer. Naming the limitation and the fix is worth more
marks than a slightly higher BLEU.

**Both checkpoints are in Drive** under `checkpoints/`, so you can reload them without retraining:

```python
ck = torch.load(f'{DRIVE}/checkpoints/ps1_lstm_best.pt')
print(ck['macro_f1'])
```